# MongoDB to Unity Catalog Migration

**Prerequisites:**
- Maven library installed on cluster: `org.mongodb.spark:mongo-spark-connector_2.13:10.4.0`
- DBR 17 LTS
- MongoDB connection URI with read access
- Unity Catalog target catalog/schema with write access

## Configuration
Update the widgets below before running the migration.

In [0]:
!pip install pymongo

In [0]:
# Configuration - Databricks Widgets
dbutils.widgets.text("mongo_uri", "mongodb+srv://<user>:<password>@<cluster>.mongodb.net", "MongoDB URI")
dbutils.widgets.text("mongo_databases", "", "MongoDB Databases (comma-separated)")
dbutils.widgets.text("collections", "", "Collections (comma-separated, leave empty for all)")
dbutils.widgets.text("target_catalog", "", "UC Target Catalog")
dbutils.widgets.dropdown("write_mode", "overwrite", ["overwrite", "append"], "Write Mode")

In [0]:
# Read widget values
mongo_uri = dbutils.widgets.get("mongo_uri")
mongo_databases_input = dbutils.widgets.get("mongo_databases")
collections_input = dbutils.widgets.get("collections")
target_catalog = dbutils.widgets.get("target_catalog")
write_mode = dbutils.widgets.get("write_mode")

# Parse databases list
mongo_databases = [db.strip() for db in mongo_databases_input.split(",") if db.strip()]

# Parse collections list (applied to all databases; leave empty to auto-discover per DB)
if collections_input.strip():
    collections_filter = [c.strip() for c in collections_input.split(",")]
else:
    collections_filter = None  # Will discover all collections per database

print(f"MongoDB URI: {mongo_uri.split('@')[-1] if '@' in mongo_uri else '***'}")
print(f"MongoDB Databases: {mongo_databases}")
print(f"Collections filter: {collections_filter or 'ALL (auto-discover per DB)'}")
print(f"Target Catalog: {target_catalog}")
print(f"Strategy: 1 UC schema per MongoDB database")
print(f"Write Mode: {write_mode}")

In [0]:
def list_mongo_collections(uri: str, database: str) -> list:
    """
    Discover all collections in the specified MongoDB database.
    Uses the MongoDB Spark Connector to read the listCollections command.
    """
    # Read a dummy collection to get access - we'll use the admin command approach
    from pymongo import MongoClient
    
    client = MongoClient(uri)
    db = client[database]
    collections = db.list_collection_names()
    client.close()
    
    # Filter out system collections
    collections = [c for c in collections if not c.startswith("system.")]
    print(f"Found {len(collections)} collections in '{database}': {collections}")
    return collections

In [0]:
# Build a mapping: database -> list of collections to migrate
db_collections_map = {}

for db_name in mongo_databases:
    if collections_filter:
        db_collections_map[db_name] = collections_filter
        print(f"[{db_name}] Using specified collections: {collections_filter}")
    else:
        discovered = list_mongo_collections(mongo_uri, db_name)
        db_collections_map[db_name] = discovered
        print(f"[{db_name}] Discovered {len(discovered)} collections")

total_collections = sum(len(v) for v in db_collections_map.values())
print(f"\nTotal: {total_collections} collections across {len(mongo_databases)} databases")

In [0]:
def read_mongo_collection(uri: str, database: str, collection: str):
    """
    Read a MongoDB collection into a Spark DataFrame using the MongoDB Spark Connector.
    """
    df = (spark.read
        .format("mongodb")
        .option("connection.uri", uri)
        .option("database", database)
        .option("collection", collection)
        .load()
    )
    return df


def sanitize_column_names(df):
    """
    Sanitize column names for Delta Lake compatibility.
    Replaces dots, spaces, and special chars with underscores.
    """
    import re
    for col_name in df.columns:
        sanitized = re.sub(r'[^a-zA-Z0-9_]', '_', col_name)
        sanitized = re.sub(r'_+', '_', sanitized).strip('_')
        if sanitized != col_name:
            df = df.withColumnRenamed(col_name, sanitized)
    return df


def write_to_unity_catalog(df, catalog: str, schema: str, table_name: str, mode: str = "overwrite"):
    """
    Write a DataFrame to Unity Catalog as a Delta table.
    """
    full_table_name = f"{catalog}.{schema}.{table_name}"

    # Sanitize column names for Delta compatibility
    df = sanitize_column_names(df)

    # Break the V2 data source lineage by going through a temp view.
    # Unity Catalog's saveAsTable does not support V2 source relations directly.
    temp_view = f"_tmp_{schema}_{table_name}"
    df.createOrReplaceTempView(temp_view)

    if mode == "overwrite":
        spark.sql(f"CREATE OR REPLACE TABLE {full_table_name} AS SELECT * FROM {temp_view}")
    else:
        spark.sql(f"INSERT INTO {full_table_name} SELECT * FROM {temp_view}")

    spark.catalog.dropTempView(temp_view)

    return full_table_name

In [0]:
# Ensure the target catalog and one schema per MongoDB database exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {target_catalog}")

for db_name in mongo_databases:
    schema_name = db_name.lower().replace("-", "_").replace(" ", "_")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{schema_name}")
    print(f"Schema ready: {target_catalog}.{schema_name} (from MongoDB database '{db_name}')")

In [0]:
from datetime import datetime

# Migration execution
results = []
failed = []

total_collections = sum(len(v) for v in db_collections_map.values())
print(f"Starting migration of {total_collections} collections across {len(db_collections_map)} databases...")
print("=" * 60)

for db_name, collections in db_collections_map.items():
    # Each MongoDB database maps to its own UC schema
    target_schema = db_name.lower().replace("-", "_").replace(" ", "_")
    print(f"\n--- Database: {db_name} -> Schema: {target_catalog}.{target_schema} ---")
    
    for collection in collections:
        start_time = datetime.now()
        print(f"  [{collection}] Reading from MongoDB...")
        
        try:
            # Read from MongoDB
            df = read_mongo_collection(mongo_uri, db_name, collection)
            row_count = df.count()
            col_count = len(df.columns)
            print(f"  [{collection}] Read {row_count:,} rows, {col_count} columns")
            
            # Write to Unity Catalog under the database-specific schema
            table_name = collection.lower().replace("-", "_").replace(" ", "_")
            full_name = write_to_unity_catalog(df, target_catalog, target_schema, table_name, write_mode)
            
            elapsed = (datetime.now() - start_time).total_seconds()
            print(f"  [{collection}] ✓ Written to {full_name} in {elapsed:.1f}s")
            
            results.append({
                "database": db_name,
                "collection": collection,
                "table": full_name,
                "rows": row_count,
                "columns": col_count,
                "duration_sec": elapsed,
                "status": "SUCCESS"
            })
            
        except Exception as e:
            elapsed = (datetime.now() - start_time).total_seconds()
            print(f"  [{collection}] ✗ FAILED: {str(e)}")
            failed.append({
                "database": db_name,
                "collection": collection,
                "error": str(e),
                "duration_sec": elapsed,
                "status": "FAILED"
            })

print("\n" + "=" * 60)
print(f"Migration complete: {len(results)} succeeded, {len(failed)} failed")

In [0]:
# Migration Summary Report
import pandas as pd

all_results = results + failed
if all_results:
    summary_df = pd.DataFrame(all_results)
    display(summary_df)
else:
    print("No migration results to display.")